# Silver Layer - Exploratory Data Analysis

This notebook explores the Bronze datasets from CRM and ERP to understand their structure, data quality and relationships before implementing Silver transformations.

The findings will guide decisions on data cleansing, standardisation, validation and integration in the Silver layer.

In [0]:
SELECT 
cst_id,
COUNT(*)
FROM sales.bronze.crm_cust_info
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL

In [0]:
SELECT 
*
FROM (
SELECT 
*,
ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last
FROM sales.bronze.crm_cust_info ) t WHERE flag_last = 1 ;

In [0]:
-- Check unwnated spaces
-- Expectation: No Results



SELECT cst_firstname
FROM sales.bronze.crm_cust_info
WHERE cst_firstname != TRIM(cst_firstname)


In [0]:
SELECT prd_nm
FROM sales.bronze.crm_prd_info
WHERE prd_nm != TRIM(prd_nm)

In [0]:
SELECT prd_cost
FROM sales.bronze.crm_prd_info
WHERE prd_cost < 0 OR prd_cost IS NULL

In [0]:
SELECT 
cst_id,
cst_key,
TRIM(cst_firstname),
TRIM(cst_lastname),
cst_marital_status,
cst_gndr,
cst_create_date
FROM (
SELECT 
*,
ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last
FROM sales.bronze.crm_cust_info ) t WHERE flag_last = 1 ;

In [0]:
-- Data Standardication & Consistancy 

SELECT DISTINCT cst_gndr
FROM sales.bronze.crm_cust_info;

In [0]:
SELECT 
cst_id,
cst_key,
TRIM(cst_firstname),
TRIM(cst_lastname),
cst_marital_status,
CASE WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
     WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
     ELSE 'n/a'
END
cst_create_date
FROM (
SELECT 
*,
ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last
FROM sales.bronze.crm_cust_info ) t WHERE flag_last = 1 ;

In [0]:
-- Data Standardication & Consistancy 

SELECT DISTINCT cst_marital_status
FROM sales.bronze.crm_cust_info;

In [0]:
INSERT INTO sales.silver.crm_cust_info (
    cst_id, 
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date )



SELECT 
cst_id,
cst_key,
TRIM(cst_firstname),
TRIM(cst_lastname),
CASE WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
     WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
     ELSE 'n/a'
END cst_marital_status,
CASE WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
     WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
     ELSE 'n/a'
END cst_gndr,
cst_create_date
FROM (
SELECT 
*,
ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last
FROM sales.bronze.crm_cust_info ) t WHERE flag_last = 1 ;